# A Toy FCC-ee Flavour Tagger with Pythia and PyTorch

Written by:
* Christian Bierlich (Lund University)

Editing and solutions by:
* Philip Ilten (University of Cincinnati)

This tutorial demonstrates how to use the [PYTHIA event generator](https://pythia.org/) to generate pseudo-data for use in machine learning applications. Here we demonstrate by building simple event features, and train a small neural network to classify hadronic Z events as

$$
Z \to q\bar q,\qquad q \in [\{u,d,s\},c, b].
$$

This notebook is designed for a short live tutorial. It is intentionally truth-level and simple. This is *not* a detector-level FCC-ee flavour tagger. A real tagger would use reconstructed tracks, impact parameters, secondary vertices, jet information, detector effects, and calibration. Here we isolate the Monte-Carlo-to-ML workflow.

After this tutorial you will know:
* How to quickly install PYTHIA through PyPI and set up simple runs for FCC physics.
* How to generate events and extract particles from the event record. Also: What is an event record?
* How to use the generated particles to construct observables/features.
* How to use PYTHIA-generated features as input to a simple truth-level tagger.

Each cell includes one or two small exercises for the interested participant to complete on their own after the tutorial to engage more deeply with the material. It is not necessary to complete an exercise to proceed to the next cell and see the full demo. Completing exercises may, however, improve performance.

## Requirements

Before running this notebook, we need to set up our environment. First, we install and import the `wurlitzer` module. This allows programs that have C-like backends to write their output to the Python console. In short, this allows the output of Pythia to be displayed in this notebook.

In [ ]:
# Redirect the C output of Pythia to the notebook.
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

Next, we need to install and import the Pythia module.

In [ ]:
# Install and import the Pythia module.
!pip install pythia8mc
import pythia8mc as pythia8

###START_SOLUTION If we want to use batch operations with Pythia, we need the `awkward` and `vector` modules. We also want to time batch operations versus normal using `time`.
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# Install and import the awkward module.
!pip install awkward
import awkward as ak

# Install and import the vector module.
!pip install vector
import vector

# Register the vector module with awkward.
vector.register_awkward()

# Import the time module.
import time

###STOP_SOLUTION

We use `numpy` for fast array operations and `matplotlib` for plotting.

In [ ]:
# Import the numpy module.
import numpy as np

# Import the matplotlib module.
import matplotlib.pyplot as plt

We use `torch` for machine learning.

In [ ]:
# Import the torch module and sub-modules.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Set the torch backend.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

We also need access to a few built-in Python modules.

In [ ]:
# Import the math module for common math operations.
import math

# Import pathlib for path manipulation.
import pathlib

Finally, we set up common seed for reproducibility with random number generation (RNG).

In [ ]:
# Set the RNG seed.
SEED = 12345
# Set the seed for numpy.
np.random.seed(SEED)
# Set the seed for torch.
torch.manual_seed(SEED)

# Set up Pythia

We first create a vanilla Pythia object for
$$
e^+e^-
$$
collisions at the Z pole. The relevant physics process is:
$$
e^+e^- \to Z^0/\gamma^* \to q\bar q.
$$
For a more realistic reproduction of LEP conditions, one would discard events where photons are radiated from the initial-state electron and positron beams. This can be emulated by switching off photon radiation from initial state leptons. Go to the [Pythia manual](https://pythia.org/latest-manual/Welcome.html), find the correct setting under `Parton showers` -> `The Simple Shower` -> `Spacelike showers` and apply it.

In [ ]:
## Create a PYTHIA object. Prints a splash screen.
pythia = pythia8.Pythia()

# Read settings. This sets up the physics process to be studied.
# Settings can also be read from an external file, see pythia.org
# for more information.

# Define the beams.
# FCC-ee-like clean e+e- environment at the Z pole.
# Numbers are particle ids, see PDG for an overview
# https://pdg.lbl.gov/2024/reviews/rpp2024-rev-monte-carlo-numbering.pdf
# Relevant numbers for this tutorial are:
# electron = 11, positron = -11, Z boson = 23, quarks u,d,s,c,b = 1,2,3,4,5
pythia.readString("Beams:idA = 11")
pythia.readString("Beams:idB = -11")
pythia.readString("Beams:eCM = 91.2")

# Set the collision process from PYTHIA"s internal list of matrix
# elements. PYTHIA can also be easily interfaced to external
# matrix element, notably MadGraph (see http://madgraph.phys.ucl.ac.be/)
# e+e- -> gamma*/Z0.
pythia.readString("WeakSingleBoson:ffbar2gmZ = on")

# Force hadronic Z decays to u,d,s,c,b.
pythia.readString("23:onMode = off")
pythia.readString("23:onIfAny = 1 2 3 4 5")

# Random number seed.
pythia.readString("Random:setSeed = on")
pythia.readString(f"Random:seed = {SEED}")
###START_SOLUTION

# Turn off photon radiation from leptons.
pythia.readString("SpaceShower:QEDshowerByL = off")

###STOP_SOLUTION
# Initialize PYTHIA before generating events.
# This prints list of process initialization and changed settings.
# The init call returns a bool for successful initialization.
if not pythia.init():
    print("Initialization failed!")

# Generate and Event

Once PYTHIA is set up to run a specific process, events are generated with successive calls to `pythia.next()`. Each such call produces an event record, which is a list of particles with properties such as status, ID, momenta, *etc.* Here we generate just a single event, and inspect the event record. In the next cell we loop over all particles in the event record and print their ID and their transverse momentum.

In [ ]:
# Generate a single event. Returns a boolean for successful generation
if not pythia.next():
    print("Event generation failed")

# Inspect the event record
# The event record is automatically printed for the first event
# It can always be printed with a call to:
# pythia.event.list()

In [ ]:
# You can then loop over particles in the event and extract properties.
# Here we print species and transverse momentum of final particles.
for prt in pythia.event:
    if prt.isFinal():
        print(f"id = {prt.id()}, pT = {prt.pT()} GeV")

Try the following.
1) Generate another event, print its event record and compare the two different event records.
2) Extend the particle loop to print all components of the particle four-momentum. If the particle has `ID > 100`, also print its production vertex. Use the [online manual](https://pythia.org/latest-manual/Welcome.html), click `Study output` -> `Particle properties`. That is a list of all properties one can extract from Pythia `Particle` objects, and thus a very useful reference point.

In [ ]:
###START_SOLUTION
# Generate the next event.
pythia.next()

# Print the event.
pythia.event.list()
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# Print each particle.
for prt in pythia.event:
    if not prt.isFinal():
        continue
    # Print the momentum.
    out = f"id = {prt.id()}, p = {prt.p()}"
    # Print the vertex.
    if prt.idAbs() > 100:
        out += f"vProd = {prt.vProd()}"
    print(out)
###STOP_SOLUTION

# Three Labelled Pythia Generators

For the toy tagger we need to label events using truth information. Our goal is to make the tagger infer truth-level/latent information from Pythia. In this case whether a given Z boson decayed to light ($u, d, s$), charm ($c$) or bottom/beauty ($b$) quarks. In the experiment these labels are not known, since quarks are not observed directly.

We solve the labelling issue in a simple way: construct three labelled generators which only generate events of the given kind.
$$
\begin{aligned}
\text{label 0: }&\quad Z \to u\bar u, d\bar d, s\bar s; \\
\text{label 1: }&\quad Z \to c\bar c; \\
\text{label 2: }&\quad Z \to b\bar b. \\
\end{aligned}
$$
PYTHIA gives both events and truth labels. In the code below, photon radiation is still included. As an exercise, and using the setting you determined from above, switch this photon radiation off.

In [ ]:
# Set up a small generator class holding process names, label, the main setting
# to be changed and the generator object itself.
class ProcessGenerator:
    """
    Class to generate Pythia events, but with different configurations.
    """

    def __init__(self, process_name, label, settings, seed=31415):
        """
        Initialize the generator.

        process_name: the name for generator.
        label: numeric label for the generator.
        settings: settings used to configure the generator.
        seed: RNG seed for the generator.
        """
        # Set the process name.
        self.process_name = process_name
        # Set the label.
        self.label = label
        # Create and initialize the Pythia object.
        self.pythia = self.setup_pythia(settings, seed)

    def setup_pythia(self, settings, seed, quiet=True):
        """
        Instantiate, configure, and initialize a Pythia instance.

        settings: settings used to configure the generator.
        seed: RNG seed for the generator.
        quiet: suppress extra output from Pythia.
        """
        # Create a Pythia object. The extra argument removes the splash screen.
        pythia = pythia8.Pythia("", not quiet)
        # Turn off most of Pythia's printing to screen.
        if quiet:
            pythia.readString("Print:quiet = on")

        # Set up beams.
        pythia.readString("Beams:idA = 11")
        pythia.readString("Beams:idB = -11")
        pythia.readString("Beams:eCM = 91.2")
        ###START_SOLUTION

        # Turn of initial state photon radiation.
        pythia.readString("SpaceShower:QEDshowerByL = off")

        ###STOP_SOLUTION
        # Set up main process.
        pythia.readString("WeakSingleBoson:ffbar2gmZ = on")

        # Force the Z decay channel defining this labelled class.
        pythia.readString("23:onMode = off")
        # Read the additional user settings.
        for setting in settings:
            pythia.readString(setting)

        # Set random number seed.
        pythia.readString("Random:setSeed = on")
        pythia.readString(f"Random:seed = {seed}")

        # Initialize with given settings, check for failure.
        if not pythia.init():
            raise RuntimeError("Pythia could not initialize! Check settings.")
        return pythia

We can now create our list of generators.

In [ ]:
# The setting 23:onIfAny assumes that all decays for particle 23 (the Z) have
# been switched off. `23:onIfAny = 1 2 3` then means if a Z can decay to any of
# the particles 1, 2 or 3 (up, down, strange), switch those decays on. Beware,
# order matters when changing PYTHIA settings. The last set setting has
# precedence.
generators = [
    ProcessGenerator("light", 0, settings=["23:onIfAny = 1 2 3"]),
    ProcessGenerator("charm", 1, settings=["23:onIfAny = 4"]),
    ProcessGenerator("bottom", 2, settings=["23:onIfAny = 5"]),
]

# Physics Features

We define two blocks of features, all of the type single number per event for simplicity.

*Physics observables:* the physics observables are defined at "particle level", and are what you would see in your usual physics analysis as resulting observables.
* $N_\text{final}$: number of final particles
* $N_\text{ch}$: Number of charged final particles
* $\sum E$: sum of energies of final particles in GeV
* $\langle E \rangle$: Mean energy of finals in GeV
* $N_e$: number of electrons
* $N_\mu$: number of muons

*Displaced-vertex features:* the displaced-vertex features are truth-level proxies for displaced decay vertex information in a realistic measurement. This is very far from a realistic detector simulation, but shows how truth-level and "detector-level" information could be combined in a single feature set, had one for example run the Pythia events through a full detector simulation.
* $\text{max}(r)$: the maximal decay radius in the event in mm
* $\langle r\rangle$: mean decay radius in mm
* $N(r>r')$: counts over threshold with $r > r'$

Pythia provides particle production vertices in Cartesian coordinates, with the collision itself in $(0,0)$. In Pythia output, lengths have units mm. Pythia has several built-in features that allow the calculation of production vertices at the fm scale as well; they are all disabled by default. Should you enable them, remember to convert units.

In a normal Pythia analysis we would fill histograms. Since we need the features for training, we fill `numpy` arrays instead.

In [ ]:
# Calculate all features for one event and return as a `numpy` array.
def event_features(event):
    """
    Calculate all features for one event and return as a `numpy` array.

    event: Pythia event.
    """
    # The r1 and r2 cuts in mm.
    r1 = 0.1
    r2 = 1.0
    # Initialize statistics gathering.
    radii = []
    n_final = 0
    n_charged = 0
    sum_E = 0
    n_e = 0
    n_mu = 0
    ###START_SOLUTION
    n_K = 0
    met = 0
    ###STOP_SOLUTION
    # Loop over all particles in event.
    # You can usually extract all relevant features in a single loop.
    # rather than doing many filter operations.
    for prt in event:
        # Look only at final particles.
        if prt.isFinal():
            n_final += 1
            # Charged particle multiplicity.
            if prt.isCharged():
                n_charged += 1
            # Summed energy.
            sum_E += prt.e()
            # Number of electrons and muons.
            if abs(prt.id()) == 11:
                n_e += 1
            elif abs(prt.id()) == 13:
                n_mu += 1
            # Displacements.
            r = math.sqrt(prt.xProd() ** 2 + prt.yProd() ** 2)
            if r > 1e-10:
                radii.append(r)
            ###START_SOLUTION
            # Number of kaons.
            if prt.idAbs() in (321, 311, 310, 130):
                n_K += 1
            # Missing transverse energy (see Feature Plots for discussion).
            if prt.idAbs() in (12, 14, 16):
                met += prt.eT()
            ###STOP_SOLUTION
    # Make the feature vector with observables.
    feats = [n_final, n_charged, sum_E, (sum_E / n_final) if n_final else 0, n_e, n_mu]
    # Add the displacements.
    if radii:
        radii = np.array(radii)
        feats += [np.max(radii), np.mean(radii), np.sum(radii > r1), np.sum(radii > r2)]
    else:
        feats += [0.0, 0.0, 0.0, 0.0]
    ###START_SOLUTION
    # Add the number of kaons.
    feats += [n_K]
    # Add the missing transverse energy (see Feature Plots for discussion).
    feats += [met]
    ###STOP_SOLUTION
    return np.array(feats, dtype=np.float32)


# Define the feature names.
# Add the physics observables.
# * number of final particles
# * number of charged final particles
# * sum of energies of finals
# * mean energy of finals
# * number of electrons
# * number of muons
event_features.names = [
    r"N_\text{final}",
    r"N_\text{ch}",
    r"\sum E~[\text{GeV}]",
    r"\langle E \rangle~[\text{GeV}]",
    "N_e",
    r"N_\mu",
]
# Add the displaced-vertex features.
# * max displacement radius
# * mean displacement radius
# * sum r > r'
event_features.names += [
    r"\text{max}(r)~[\text{mm}]",
    r"\langle r\rangle~[\text{mm}]",
    "N(r>r_1)",
    "N(r>r_2)",
]
###START_SOLUTION
# Add the number of kaons.
event_features.names += ["N_K"]
# Add the missing transverse energy (see Feature Plots for discussion).
event_features.names += [r"\text{missing } E_T~[\text{GeV}]"]
###STOP_SOLUTION

In [ ]:
# Inspect one event's feature vector.
generators[0].pythia.next()
demo_feats = event_features(generators[0].pythia.event)
max_name = max(len(name) for name in event_features.names)
for name, value in zip(event_features.names, demo_feats):
    print(f"{name:>{max_name}}: {value:10.4g}")

Come up with another physics observable feature and add it above to `event_features`. For this example, features relating to strange particle production, such as number of kaons, could be relevant. You will need to look up the particle ID for kaons, add a clause in the particle loop, and fill a new feature.

# Build or Load the Dataset

We now generate labelled events and save the features. This is the slow step in the notebook, because we repeatedly ask PYTHIA to produce an event, loop over the event record, and reduce it to a small list of numbers. For a real production workflow, this is where one would think more carefully about scaling.

At FCC-ee, the projected $Z$-pole samples are enormous. One would not generate the full training sample interactively inside a notebook. The usual pattern would be to run event generation as a batch job, for example on a high-performance cluster or grid resource, and then train on the reduced output. The reduced output is not the full PYTHIA event record, but a derived analysis format: feature arrays, ROOT files, parquet/awkward arrays, or another compact representation adapted to the training task.

The current Pythia Python interface also provides a more efficient batch-generation route through Awkward Arrays. Instead of translating the C++ event record particle-by-particle into Python inside a loop, one can ask Pythia for a batch of events using `nextBatch`. This returns an Awkward array containing event information and particle information, which can then be sliced and processed in a vectorized, columnar style. That is often closer to how modern HEP analysis workflows are written.

Schematically, the simple tutorial approach is as follows.
```python
# Loop over events.
for i in range(n_events):
    # Generate the event.
    pythia.next()
    # Extract the features.
    x = event_features(pythia.event)
```
whereas a more scalable Python approach would look more like the following.
```python
# Import awkward and vector modules.
import awkward as ak
import vector
# Register the vector module with awkward.
vector.register_awkward()
events = pythia.nextBatch(n_events)
# Operate on `events.prt.id`, `events.prt.p.e`, `events.prt`, etc. using
# awkward-array slicing and reductions.
```
This is not necessary for the small example below, but it is useful to know that the notebook loop is not the only possible production version.

Pythia can also interface to standard HEP event formats. For workflows where the downstream analysis is not written directly against the Pythia event record, events can be passed through common formats such as Les Houches Event Files, HepMC, and ROOT-based outputs.

All this is documented in the [online Pythia manual](https://pythia.org/latest-manual/Welcome.html).

In [ ]:
# Load the training data.
def load_dataset(cache_file, cache_load):
    """
    Load the training data, returning `X, y, feature_names` where `X` are the
    event features, `y` are the labels, and `feature_names` are the features.

    cache_file: file path to the data cache to load.
    cache_load: if `True`, load the cache if available.
    """
    msg = "No cache, generating sample."
    if cache_file:
        cache_file = pathlib.Path(cache_file)
        if cache_load and cache_file.exists():
            data = np.load(cache_file)
            X = data["X"].astype(np.float32)
            y = data["y"].astype(np.int64)
            feature_names = list(data["feature_names"])
            print("Loaded cached dataset:", cache_file)
            print("X shape:", X.shape, "y shape:", y.shape)
            return X, y, feature_names, None
        elif cache_file.exists():
            msg = "Regenerating sample."
    return None, None, None, msg

In [ ]:
# Generate or load the training data.
def generate_dataset(
    generators,
    cache_file="pythia_fcc_ee_features.npz",
    cache_load=True,
    n_per_class=1e5,
):
    """
    Generate or load the training data, returning `X, y, feature_names` where
    `X` are the event features, `y` are the labels, and `feature_names` are the
    features.

    generators: list of generators of type type `ProcessGenerator`.
    cache_file: file path to the data cache, either to load or save. If `None`,
                do not load or save the cache.
    cache_load: if `True`, load the cache if available.
    n_per_class: number of events to generate per class.
    """
    # Load the cache if available.
    X, y, feature_names, msg = load_dataset(cache_file, cache_load)
    if msg is None:
        return X, y, feature_names
    print(msg)

    # Generate the data.
    feature_names = np.array(event_features.names)
    n_total = len(generators) * int(n_per_class)
    n_features = len(feature_names)
    X = np.empty((n_total, n_features), dtype=np.float32)
    y = np.empty(n_total, dtype=np.int64)
    i = 0
    # Loop over the generators.
    for g in generators:
        n_generated = 0
        while n_generated < n_per_class:
            # Generate the event.
            if g.pythia.next():
                # Extract the event features.
                X[i] = event_features(g.pythia.event)
                y[i] = g.label
                i += 1
                n_generated += 1

    # Shuffle once so classes are not contiguous.
    perm = np.random.default_rng(SEED).permutation(len(y))
    X = X[perm]
    y = y[perm]

    # Save the cache if requested.
    if cache_file:
        np.savez(cache_file, X=X, y=y, feature_names=feature_names)
        print("Saved cache:", cache_file)

    # Return the data.
    print("X shape:", X.shape, "y shape:", y.shape)
    return X, y, feature_names

In [ ]:
###START_PROBLEM
# Generate the dataset.
X, y, feature_names = generate_dataset(generators, n_per_class=1e4)
# Print the result.
print("Class counts:")
for k in [g.label for g in generators]:
    print(f" {np.sum(y == k)}")
###STOP_PROBLEM

Update the generation above to use batch generation. Go to the Pythia online documentation, and find the example `main297.py` which demonstrates the use of batch generation. Modify the `generate_dataset` above, or make a new one, where you take both approaches. Time the difference.

In [ ]:
###START_SOLUTION
# First, we define an operator to create the features for a batch of events.
def batch_event_features(events):
    """
    Calculate all features for an array of events and return as a
    `numpy` array.

    events: Awkward array of Pythia events.
    """
    # The r1 and r2 cuts in mm.
    r1 = 0.1
    r2 = 1.0

    # Get only final state particles.
    final = events.prt[events.prt.status > 0]
    # Particle IDs.
    id_abs = abs(final.id)

    # Number of final state particles.
    n_final = ak.to_numpy(ak.num(final, axis=1)).astype(np.float32)
    # Number of charges particles (we do this by negation).
    n_charged = ak.to_numpy(
        ak.sum(
            ~(
                (id_abs == 22)
                | (id_abs == 12)
                | (id_abs == 14)
                | (id_abs == 16)
                | (id_abs == 130)
                | (id_abs == 2112)
            ),
            axis=1,
        )
    ).astype(np.float32)
    # Summed energy.
    sum_E = ak.to_numpy(ak.sum(final.p.e, axis=1)).astype(np.float32)
    # Mean energy.
    mean_E = np.divide(sum_E, n_final, out=np.zeros_like(sum_E), where=n_final > 0)
    # Number of electrons.
    n_e = ak.to_numpy(ak.sum(id_abs == 11, axis=1)).astype(np.float32)
    # Number of muons.
    n_mu = ak.to_numpy(ak.sum(id_abs == 13, axis=1)).astype(np.float32)

    # Displacements. Filter out particles with no production vertex.
    disp = final[~ak.is_none(final.vProd, axis=1)]
    r = np.sqrt(disp.vProd.x**2 + disp.vProd.y**2)
    r = r[r > 1e-10]
    # The following behaviour fills with 0 if there are no displaced particles.
    max_r = ak.to_numpy(ak.fill_none(ak.max(r, axis=1), 0)).astype(np.float32)
    r_sum = ak.to_numpy(ak.sum(r, axis=1)).astype(np.float32)
    r_count = ak.to_numpy(ak.num(r, axis=1)).astype(np.float32)
    mean_r = np.divide(r_sum, r_count, out=np.zeros_like(r_sum), where=r_count > 0)
    n_r1 = ak.to_numpy(ak.sum(r > r1, axis=1)).astype(np.float32)
    n_r2 = ak.to_numpy(ak.sum(r > r2, axis=1)).astype(np.float32)

    # Number of kaons.
    n_K = ak.to_numpy(
        ak.sum(
            (id_abs == 321) | (id_abs == 311) | (id_abs == 310) | (id_abs == 130),
            axis=1,
        )
    ).astype(np.float32)
    # Missing energy.
    met = ak.to_numpy(
        ak.sum(final.p.et[(id_abs == 12) | (id_abs == 14) | (id_abs == 16)], axis=1)
    ).astype(np.float32)

    # Return the features.
    return np.column_stack(
        [
            n_final,
            n_charged,
            sum_E,
            mean_E,
            n_e,
            n_mu,
            max_r,
            mean_r,
            n_r1,
            n_r2,
            n_K,
            met,
        ]
    ).astype(np.float32)


# Define the feature names.
batch_event_features.names = event_features.names
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# Second, we create the batch generation function.
def batch_generate_dataset(
    generators,
    cache_file="batch_pythia_fcc_ee_features.npz",
    cache_load=True,
    n_per_class=1e5,
):
    """
    Batch generate or load the training data, returning `X, y, feature_names`
    where `X` are the event features, `y` are the labels, and `feature_names`
    are the features.

    generators: list of generators of type type `ProcessGenerator`.
    cache_file: file path to the data cache, either to load or save. If `None`,
                do not load or save the cache.
    cache_load: if `True`, load the cache if available.
    n_per_class: number of events to generate per class.
    """
    # Load the cache if available.
    X, y, feature_names, msg = load_dataset(cache_file, cache_load)
    if msg is None:
        return X, y, feature_names
    print(msg)

    # Generate the data.
    feature_names = np.array(event_features.names)
    n_per_class = int(n_per_class)
    n_total = len(generators) * int(n_per_class)
    n_features = len(feature_names)
    X = np.empty((n_total, n_features), dtype=np.float32)
    y = np.empty(n_total, dtype=np.int64)
    i = 0
    # Loop over the generators.
    for g in generators:
        # Generate the events.
        events = g.pythia.nextBatch(n_per_class)
        # Extract the event features.
        X[i : i + n_per_class] = batch_event_features(events)
        y[i : i + n_per_class] = g.label
        i += n_per_class

    # Shuffle once so classes are not contiguous.
    perm = np.random.default_rng(SEED).permutation(len(y))
    X = X[perm]
    y = y[perm]

    # Save the cache if requested.
    if cache_file:
        np.savez(cache_file, X=X, y=y, feature_names=feature_names)
        print("Saved cache:", cache_file)

    # Return the data.
    print("X shape:", X.shape, "y shape:", y.shape)
    return X, y, feature_names


###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# Batch generate the dataset.
X, y, feature_names = batch_generate_dataset(
    generators, cache_file=None, n_per_class=1e3
)
# Print the result.
print("Class counts:")
for k in [g.label for g in generators]:
    print(f" {np.sum(y == k)}")
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# We can now test the timing.
# Time batch first.
batch_start = time.perf_counter()
X, y, feature_names = batch_generate_dataset(
    generators, cache_file=None, n_per_class=1e3
)
batch_time = time.perf_counter() - batch_start

# Time serial next.
serial_start = time.perf_counter()
X, y, feature_names = generate_dataset(generators, cache_file=None, n_per_class=1e3)
serial_time = time.perf_counter() - serial_start

# Print the results.
print(f"batch time: {batch_time:.4f} s")
print(f"serial time: {serial_time:.4f} s")
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
# Generate the dataset.
X, y, feature_names = batch_generate_dataset(generators, n_per_class=1e4)
# Print the result.
print("Class counts:")
for k in [g.label for g in generators]:
    print(f" {np.sum(y == k)}")
###STOP_SOLUTION

# Feature Plots

We inspect a few features before giving the truth simulation to a neural network.

Inspect the features. Are there any physics features which are weirdly defined, such that they will not add any discriminating power to the network? (spoiler: there are). Find that feature, go back and check its definition. Figure out what similar feature to replace it with, implement the change and regenerate the training data.

###START_SOLUTION The summed energy (still shown below), will always return the $Z$-pole mass, regardless of the final state, and so does not provide any discriminating power. Instead, the summed transverse energy could be different, but even more discriminating is the summed missing transverse energy (MET).
###STOP_SOLUTION

In [ ]:
# Construct simple histograms of all generated features.
# Loop over the features.
for name in event_features.names:
    plt.figure()
    # Loop over the generator labels.
    for g in generators:
        # Plot the histogram for this label.
        plt.hist(
            X[y == g.label, event_features.names.index(name)],
            histtype="step",
            density=True,
            label=str(g.process_name),
        )
    # Format the plot.
    plt.xlabel(f"${name}$")
    plt.ylabel("normalized events")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Train a Small Classifier

At this point PYTHIA has done the important job: it has turned physics settings into a table of features `X` and truth labels `y`. We make a two-way 75/25 split in a training and testing sample, since we are anyway not spending a lot of time validating the network in this tutorial. In a more realistic application this step would be more deliberately set up.

We use a boring, ordinary neural network. The point is not the architecture. The point is that the labels came for free from the generator, and the features were built from the event record.

The outputs from this frame are the two training curves: loss and accuracy versus epoch. We train for a fixed number of epochs.

In [ ]:
# Split and standardize the Pythia training data.
# Set up a reproducible random number generator and shuffle sample indices.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(y))
# Reserve 25% of the data for testing, the rest for training.
n_test = len(y) // 4
idx_test = indices[:n_test]
idx_train = indices[n_test:]

# Compute per-feature mean and std from the training set only,
# to avoid leaking test-set information into the standardization.
mean = X[idx_train].mean(axis=0)
std = X[idx_train].std(axis=0)
std[std == 0] = 1

# Standardize both splits using training statistics, cast to float32 for torch.
X_train = ((X[idx_train] - mean) / std).astype(np.float32)
X_test = ((X[idx_test] - mean) / std).astype(np.float32)
y_train = y[idx_train]
y_test = y[idx_test]

# Define torch data loaders.
# Training loader shuffles each epoch to avoid ordering effects during
# stochastic gradient descent.
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=1024,
    shuffle=True,
)
# Test loader, no shuffling needed since we're only evaluating;
# larger batch size is fine since no gradients are stored.
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=4096,
    shuffle=False,
)

# A deliberately boring network: the physics is in PYTHIA and the features.
# Simple MLP: input -> 32 -> 32 -> 3 output classes (logits not probabilities).
model = nn.Sequential(
    nn.Linear(X.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
).to(device)

# Use cross entropy for loss function and Adam for optimizer.
# CrossEntropyLoss expects raw logits and integer class labels.
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Track per-epoch metrics for later plotting.
EPOCHS = 12
train_loss, test_loss = [], []
train_acc, test_acc = [], []

# Training loop. Here we here train for a fixed number of epochs.
for epoch in range(EPOCHS):
    # Enable training-mode behavior (relevant if using dropout/batchnorm).
    model.train()
    loss_sum, n_correct, n_seen = 0.0, 0, 0

    # Training.
    for xb, yb in train_loader:
        # Move batch data to the same device as the model.
        xb, yb = xb.to(device), yb.to(device)
        # Clear gradients from previous step.
        optimizer.zero_grad()
        # Forward pass.
        logits = model(xb)
        # Compute batch loss.
        loss = loss_fn(logits, yb)
        # Back propagate.
        loss.backward()
        # Update the weights.
        optimizer.step()

        # Accumulate weighted loss and correct-prediction counts for
        # epoch-level stats.
        loss_sum += loss.item() * len(yb)
        n_correct += (logits.argmax(1) == yb).sum().item()
        n_seen += len(yb)

    # Average loss and accuracy over all training samples seen this epoch.
    train_loss.append(loss_sum / n_seen)
    train_acc.append(n_correct / n_seen)

    # Evaluation.
    # Disable training-mode behavior for evaluation.
    model.eval()
    loss_sum, n_correct, n_seen = 0.0, 0, 0
    # No need to track gradients during evaluation.
    with torch.no_grad():
        for xb, yb in test_loader:
            # Move batch data to the same device as the model.
            xb, yb = xb.to(device), yb.to(device)
            # Forward pass only.
            logits = model(xb)
            # Compute loss for this batch data.
            loss = loss_fn(logits, yb)
            # Weight the batch data loss by size before summing.
            loss_sum += loss.item() * len(yb)
            # Compare to correct labels.
            n_correct += (logits.argmax(1) == yb).sum().item()
            # Number of samples processed.
            n_seen += len(yb)

    # Convert running totals into per-sample averages for this epoch.
    test_loss.append(loss_sum / n_seen)
    test_acc.append(n_correct / n_seen)

epochs = np.arange(1, EPOCHS + 1)

# Plot and show loss and accuracy vs epochs trained.
# Loss curve, compare train vs test cross-entropy loss over training
plt.figure()
plt.plot(epochs, train_loss, marker="o", label="train")
plt.plot(epochs, test_loss, marker="o", label="test")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.tight_layout()
plt.show()
# Accuracy curve, compare train vs. test classification accuracy over training.
plt.figure()
plt.plot(epochs, train_acc, marker="o", label="train")
plt.plot(epochs, test_acc, marker="o", label="test")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.tight_layout()
plt.show()

Try some of the following.
1. Exclude features relating to displacement radii from the training (note that they were deliberately named separately above). Observe what happens to the training curves.
2. Change from a fixed number to epochs to a dynamic stopping criterion which will stop the training once the loss stops falling.
3. You can also play with learning rate, network architecture, add a dropout *etc.* at this point. The example is small enough that training will be quick, once training data has been generated once and for all.

# Validate the Classifier

The validation question is simple: after training on PYTHIA-generated features, can the network separate light, charm and bottom events on held-out events?

This validation step is kept compact, since we are now essentially done with the tutorial: We have shown how one can easily use PYTHIA to generate labelled pseudo-data and interface to torch. Therefore only two validation plots are shown: a confusion matrix and a one-vs-rest ROC curve.

For the confusion matrix we do not choose a probability threshold. Each event is assigned to the class with the largest network output score. The ROC curves below instead scan a threshold on each class probability.

In [ ]:
# Switch to evaluation mode.
model.eval()
# Disable gradient tracking since we're not training.
with torch.no_grad():
    # Run the model on the test features and move predictions to device.
    logits = model(torch.tensor(X_test).to(device))
    # Convert raw logits to class probabilities (softmax over classes),
    # then move back to CPU as a numpy array for analysis.
    probs = torch.softmax(logits, dim=1).cpu().numpy()

# Predicted class is the index of the highest probablity for each sample.
pred = probs.argmax(axis=1)
# Get the labels from the generators.
labels = [g.process_name for g in generators]

# Calculate the confusion matrix.
cm = np.zeros((3, 3), dtype=int)
for t, p in zip(y_test, pred):
    cm[t, p] += 1
# Normalize each row so that it sums to 1.
cm_norm = cm / cm.sum(axis=1, keepdims=True)

# Plot the normalized confusion matrix as a heat map.
plt.figure()
plt.imshow(cm_norm)
plt.xticks(range(3), labels)
plt.yticks(range(3), labels)
plt.xlabel("predicted")
plt.ylabel("true")
plt.colorbar(label="fraction of true class")
# # Annotate each cell with its numeric value.
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()

# Draw the receiver operating characteristic (ROC) curve.
plt.figure()
for k, label in enumerate(labels):
    # Ground truth.
    truth = y_test == k
    # Sort samples by predicted probability of class k, descending,
    # so we can sweep the decision threshold from most to least confident.
    order = np.argsort(probs[:, k])[::-1]
    truth = truth[order]
    # True positive rate.
    tpr = np.r_[0, np.cumsum(truth) / np.sum(truth)]
    # False positive rate.
    fpr = np.r_[0, np.cumsum(~truth) / np.sum(~truth)]
    # Area under the curve, using Simpson's rule.
    auc = np.trapezoid(tpr, fpr)
    plt.plot(fpr, tpr, label=f"{label}, AUC = {auc:.3f}")

# Reference line for random guess.
plt.plot([0, 1], [0, 1], linestyle="--", label="random guess")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.legend()
plt.tight_layout()
plt.show()

Make a plot of purity vs threshold, with purity defined as:
$$
P = \frac{\text{true positives}}{\text{true positives} + \text{false positives}}
$$
at a given threshold for each class, one-vs-rest. Use the purity curve to decide threshold cuts for 70, 80 and 90% purity for light, charm, and bottom respectively.

In [ ]:
###START_SOLUTION
# Define the target purities, and corresponding cuts.
target_purity = {"light": 0.70, "charm": 0.80, "bottom": 0.90}
threshold_cuts = [1, 1, 1]

# Loop over the labels.
for k, label in enumerate(labels):
    # Ground truth for this class.
    truth = y_test == k
    # Sort samples by predicted probability of class k, descending.
    order = np.argsort(probs[:, k])[::-1]
    sorted_probs = probs[order, k]
    sorted_truth = truth[order]
    # Cumulative true positives and total predicted positives at each threshold.
    tp = np.cumsum(sorted_truth)
    n_pred = np.arange(1, len(sorted_truth) + 1)
    purity = tp / n_pred
    # Plot.
    plt.plot(sorted_probs, purity, label=label)
    # Find the loosest threshold (highest efficiency).
    meets_target = purity >= target_purity[label]
    if meets_target.any():
        threshold_cuts[k] = sorted_probs[np.where(meets_target)[0][-1]]

# Label the plot.
plt.xlabel("decision threshold")
plt.ylabel("purity (precision)")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

# Print the thresholds.
for k, (label, target) in enumerate(target_purity.items()):
    thr = threshold_cuts[k]
    print(f"{label}: threshold >= {thr:.3f} for >= {target:.0%} purity")
###STOP_SOLUTION